In [50]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [51]:
import os
os.chdir(r'C:\Kaggle_Competition\Playground\S6E5-F1-Pit-Stops')
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import optuna
from scipy.stats import rankdata

# Externals
from src.utils import *
import config
import warnings
warnings.filterwarnings('ignore')

In [52]:
train = read_csv_file('data/raw/train.csv')
test = read_csv_file('data/raw/test.csv')
orig = read_csv_file('data/orig/f1_strategy_dataset_v4.csv')
orig = orig.dropna(axis=0)

train = reduce_mem_usage(train)
test = reduce_mem_usage(test)
orig = reduce_mem_usage(orig)

train_ids = train[config.ID_COL].values
test_ids = test[config.ID_COL].values

print(f'Shape of train data: {train.shape}')
print(f'Shape of test data: {test.shape}')
print(f'Shape of orig data: {orig.shape}')

Shape of train data: (439140, 16)
Shape of test data: (188165, 15)
Shape of orig data: (101305, 16)


In [53]:
# Data
train_features = train.drop(['id', config.TARGET], axis=1)
target = train[config.TARGET].values
X_test = test.drop('id', axis=1)

orig_features = orig.drop([config.TARGET], axis=1)
orig_features = orig_features.dropna(axis=0)

### Blending

In [54]:
# HistGBM
oof_proba_hist = read_csv_file(r'artifacts\oof\HISTGBM_v2_oof_proba.csv')
oof_proba_hist = oof_proba_hist.iloc[:, 1:]

test_proba_hist = read_csv_file(r'artifacts\test_proba\HISTGBM_v2_test_proba.csv')
test_proba_hist = test_proba_hist.iloc[:, 1:]

# XGBM
oof_proba_xgbm = read_csv_file(r'artifacts\oof\XGB_v9_seed42_oof_proba.csv')
oof_proba_xgbm = oof_proba_xgbm.iloc[:, 1:]

test_proba_xgbm = read_csv_file(r'artifacts\test_proba\XGB_v9_seed42_test_proba.csv')
test_proba_xgbm = test_proba_xgbm.iloc[:, 1:]

# LGBM
oof_proba_lgbm = read_csv_file(r'artifacts\oof\LGBM_v4_seed42_oof_proba.csv')
oof_proba_lgbm = oof_proba_lgbm.iloc[:, 1:]

test_proba_lgbm = read_csv_file(r'artifacts\test_proba\LGBM_v4_seed42_test_proba.csv')
test_proba_lgbm = test_proba_lgbm.iloc[:, 1:]

# ResNet
oof_proba_resnet = read_csv_file(r'artifacts\oof\RESNET_v1_seed42_oof_proba.csv')
oof_proba_resnet = oof_proba_resnet.iloc[:, 1:]

test_proba_resnet = read_csv_file(r'artifacts\test_proba\RESNET_v1_seed42_test_proba.csv')
test_proba_resnet = test_proba_resnet.iloc[:, 1:]

# REALMLP
oof_proba_realmlp = read_csv_file(r'artifacts\oof\REALMLP_v1_seed42_oof_proba.csv')
oof_proba_realmlp = oof_proba_realmlp.iloc[:, 1:]

test_proba_realmlp = read_csv_file(r'artifacts\test_proba\REALMLP_v1_seed42_test_proba.csv')
test_proba_realmlp = test_proba_realmlp.iloc[:, 1:]

In [55]:
# OOF
oof_hist = oof_proba_hist.values
oof_xgbm = oof_proba_xgbm.values
oof_lgbm = oof_proba_lgbm.values
oof_resnet = oof_proba_resnet.values
oof_realmlp = oof_proba_realmlp.values

# TEST
test_hist = test_proba_hist.values
test_xgbm = test_proba_xgbm.values
test_lgbm = test_proba_lgbm.values
test_resnet = test_proba_resnet.values
test_realmlp = test_proba_realmlp.values

In [56]:
score_hist = roc_auc_score(target, oof_hist)
score_xgbm = roc_auc_score(target, oof_xgbm)
score_lgbm = roc_auc_score(target, oof_lgbm)
score_resnet = roc_auc_score(target, oof_resnet)
score_realmlp = roc_auc_score(target, oof_realmlp)

print(f"HISTGBM alone: {score_hist:.6f}")
print(f"XGBM alone: {score_xgbm:.6f}")
print(f"LGBM alone: {score_lgbm:.6f}")
print(f"RESNET alone: {score_resnet:.6f}")
print(f"REALMLP alone: {score_realmlp:.6f}")

HISTGBM alone: 0.945228
XGBM alone: 0.948929
LGBM alone: 0.948235
RESNET alone: 0.942910
REALMLP alone: 0.954034


### Weighted Blend

In [36]:
def objective(trial):
    w_hist = trial.suggest_float("w_hist", 0.0, 1.0)
    w_xgbm = trial.suggest_float("w_xgbm", 0.0, 1.0)
    w_lgbm = trial.suggest_float("w_lgbm", 0.0, 1.0)
    w_resnet = trial.suggest_float("w_resnet", 0.0, 1.0)
    w_realmlp = trial.suggest_float("w_realmlp", 0.0, 1.0)

    total = (
        w_hist + w_xgbm + w_lgbm + w_resnet + w_realmlp
    )

    w_hist /= total
    w_xgbm /= total
    w_lgbm /= total
    w_resnet /= total
    w_realmlp /= total

    preds = (
        w_hist * oof_hist +
        w_xgbm * oof_xgbm +
        w_lgbm * oof_lgbm +
        w_resnet * oof_resnet +
        w_realmlp * oof_realmlp
    )

    return roc_auc_score(target, preds)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=5000, show_progress_bar=True)

# -------- BEST WEIGHTS --------
best = study.best_params

total = (
    best["w_hist"] +
    best["w_xgbm"] +
    best["w_lgbm"] +
    best["w_resnet"] +
    best["w_realmlp"]
)

w_hist = best["w_hist"] / total
w_xgbm = best["w_xgbm"] / total
w_lgbm = best["w_lgbm"] / total
w_resnet = best["w_resnet"] / total
w_realmlp = best["w_realmlp"] / total

# -------- PRINT --------
best_single = max(
    score_hist,
    score_xgbm,
    score_lgbm,
    score_resnet,
    score_realmlp
)

print(f"\n{'='*40}")
print(f"HISTGBM alone: {score_hist:.6f}")
print(f"XGBM alone: {score_xgbm:.6f}")
print(f"LGBM alone: {score_lgbm:.6f}")
print(f"RESNET alone: {score_resnet:.6f}")
print(f"REALMLP alone: {score_realmlp:.6f}")

print(f"Best blend: {study.best_value:.6f}")
print(f"Gain: +{study.best_value - best_single:.6f}")
print(f"{'='*40}")

print(f"HIST weight: {w_hist:.4f} ({w_hist:.1%})")
print(f"XGBM weight: {w_xgbm:.4f} ({w_xgbm:.1%})")
print(f"LGBM weight: {w_lgbm:.4f} ({w_lgbm:.1%})")
print(f"RESNET weight: {w_resnet:.4f} ({w_resnet:.1%})")
print(f"REALMLP weight: {w_realmlp:.4f} ({w_realmlp:.1%})")

[I 2026-05-13 23:41:25,027] A new study created in memory with name: no-name-9526d4fd-e763-46b2-ad65-5e59bdfc4d52


  0%|          | 0/5000 [00:00<?, ?it/s]

[I 2026-05-13 23:41:25,180] Trial 0 finished with value: 0.9501070777973033 and parameters: {'w_hist': 0.3745401188473625, 'w_xgbm': 0.9507143064099162, 'w_lgbm': 0.7319939418114051, 'w_resnet': 0.5986584841970366, 'w_realmlp': 0.15601864044243652}. Best is trial 0 with value: 0.9501070777973033.
[I 2026-05-13 23:41:25,286] Trial 1 finished with value: 0.9521216574851931 and parameters: {'w_hist': 0.15599452033620265, 'w_xgbm': 0.05808361216819946, 'w_lgbm': 0.8661761457749352, 'w_resnet': 0.6011150117432088, 'w_realmlp': 0.7080725777960455}. Best is trial 1 with value: 0.9521216574851931.
[I 2026-05-13 23:41:25,395] Trial 2 finished with value: 0.9507484110404479 and parameters: {'w_hist': 0.020584494295802447, 'w_xgbm': 0.9699098521619943, 'w_lgbm': 0.8324426408004217, 'w_resnet': 0.21233911067827616, 'w_realmlp': 0.18182496720710062}. Best is trial 1 with value: 0.9521216574851931.
[I 2026-05-13 23:41:25,524] Trial 3 finished with value: 0.9511755797063421 and parameters: {'w_hist':

In [49]:
blend_oof = (
    w_hist * oof_hist +
    w_xgbm * oof_xgbm +
    w_lgbm * oof_lgbm +
    w_resnet * oof_resnet +
    w_realmlp * oof_realmlp
)

blend_test = (
    w_hist * test_hist +
    w_xgbm * test_xgbm +
    w_lgbm * test_lgbm +
    w_resnet * test_resnet +
    w_realmlp * test_realmlp
)

# -------- SAVE BLENDED FILES --------
blend_oof_df = pd.DataFrame(blend_oof, columns=['PitNextLap'])
blend_oof_df.insert(0, "id", train_ids)

save_csv_file(
    blend_oof_df,
    os.path.join(
        config.OOF_PROBA_DIR,
        'weightblend_histV2_xgbV9_lgbV4_resnetV1_realmlpV1_oof_proba[0.954406].csv'
    )
)

blend_test_df = pd.DataFrame(blend_test, columns=['PitNextLap'])
blend_test_df.insert(0, "id", test_ids)

save_csv_file(
    blend_test_df,
    os.path.join(
        config.TEST_PROBA_DIR,
        'weightblend_histV2_xgbV9_lgbV4_resnetV1_realmlpV1_test_proba[0.954406].csv'
    )
)

### Weighted Ranked Blend

In [40]:
def objective(trial):
    w_hist = trial.suggest_float("w_hist", 0.0, 1.0)
    w_xgbm = trial.suggest_float("w_xgbm", 0.0, 1.0)
    w_lgbm = trial.suggest_float("w_lgbm", 0.0, 1.0)
    w_resnet = trial.suggest_float("w_resnet", 0.0, 1.0)
    w_realmlp = trial.suggest_float("w_realmlp", 0.0, 1.0)

    total = (
        w_hist +
        w_xgbm +
        w_lgbm +
        w_resnet +
        w_realmlp
    )

    # Normalize weights [0, 1]
    w_hist /= total
    w_xgbm /= total
    w_lgbm /= total
    w_resnet /= total
    w_realmlp /= total

    preds = (
        w_hist * rankdata(oof_hist.ravel()) +
        w_xgbm * rankdata(oof_xgbm.ravel()) +
        w_lgbm * rankdata(oof_lgbm.ravel()) +
        w_resnet * rankdata(oof_resnet.ravel()) +
        w_realmlp * rankdata(oof_realmlp.ravel())
    )

    return roc_auc_score(target, preds)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=5000,
    show_progress_bar=True
)

# -------- BEST WEIGHTS --------
best = study.best_params

total = (
    best["w_hist"] +
    best["w_xgbm"] +
    best["w_lgbm"] +
    best["w_resnet"] +
    best["w_realmlp"]
)

w_hist = best["w_hist"] / total
w_xgbm = best["w_xgbm"] / total
w_lgbm = best["w_lgbm"] / total
w_resnet = best["w_resnet"] / total
w_realmlp = best["w_realmlp"] / total

# -------- FINAL RANKED BLEND --------
blend_oof = (
    w_hist * rankdata(oof_hist.ravel()) +
    w_xgbm * rankdata(oof_xgbm.ravel()) +
    w_lgbm * rankdata(oof_lgbm.ravel()) +
    w_resnet * rankdata(oof_resnet.ravel()) +
    w_realmlp * rankdata(oof_realmlp.ravel())
)

blend_test = (
    w_hist * rankdata(test_hist.ravel()) +
    w_xgbm * rankdata(test_xgbm.ravel()) +
    w_lgbm * rankdata(test_lgbm.ravel()) +
    w_resnet * rankdata(test_resnet.ravel()) +
    w_realmlp * rankdata(test_realmlp.ravel())
)

# -------- SCORES --------
best_single = max(
    score_hist,
    score_xgbm,
    score_lgbm,
    score_resnet,
    score_realmlp
)

print(f"\n{'='*40}")
print(f"HISTGBM alone: {score_hist:.6f}")
print(f"XGBM alone: {score_xgbm:.6f}")
print(f"LGBM alone: {score_lgbm:.6f}")
print(f"RESNET alone: {score_resnet:.6f}")
print(f"REALMLP alone: {score_realmlp:.6f}")

print(f"\nBest Ranked Blend: {study.best_value:.6f}")
print(f"Gain: +{study.best_value - best_single:.6f}")
print(f"{'='*40}")

print(f"HIST weight: {w_hist:.4f} ({w_hist:.1%})")
print(f"XGBM weight: {w_xgbm:.4f} ({w_xgbm:.1%})")
print(f"LGBM weight: {w_lgbm:.4f} ({w_lgbm:.1%})")
print(f"RESNET weight: {w_resnet:.4f} ({w_resnet:.1%})")
print(f"REALMLP weight: {w_realmlp:.4f} ({w_realmlp:.1%})")

[I 2026-05-13 23:58:36,381] A new study created in memory with name: no-name-c3764324-044a-4cd0-810d-4581bbf4f2d9


  0%|          | 0/5000 [00:00<?, ?it/s]

[I 2026-05-13 23:58:36,808] Trial 0 finished with value: 0.9499864918394233 and parameters: {'w_hist': 0.3745401188473625, 'w_xgbm': 0.9507143064099162, 'w_lgbm': 0.7319939418114051, 'w_resnet': 0.5986584841970366, 'w_realmlp': 0.15601864044243652}. Best is trial 0 with value: 0.9499864918394233.
[I 2026-05-13 23:58:37,261] Trial 1 finished with value: 0.9519026560421855 and parameters: {'w_hist': 0.15599452033620265, 'w_xgbm': 0.05808361216819946, 'w_lgbm': 0.8661761457749352, 'w_resnet': 0.6011150117432088, 'w_realmlp': 0.7080725777960455}. Best is trial 1 with value: 0.9519026560421855.
[I 2026-05-13 23:58:37,688] Trial 2 finished with value: 0.950671896255953 and parameters: {'w_hist': 0.020584494295802447, 'w_xgbm': 0.9699098521619943, 'w_lgbm': 0.8324426408004217, 'w_resnet': 0.21233911067827616, 'w_realmlp': 0.18182496720710062}. Best is trial 1 with value: 0.9519026560421855.
[I 2026-05-13 23:58:38,141] Trial 3 finished with value: 0.9509661578413863 and parameters: {'w_hist': 

KeyboardInterrupt: 

### Simple ranked average

In [39]:
# OOF ranked average
blend_oof = (
    rankdata(oof_hist.ravel()) +
    rankdata(oof_xgbm.ravel()) +
    rankdata(oof_lgbm.ravel()) +
    rankdata(oof_resnet.ravel()) +
    rankdata(oof_realmlp.ravel())
) / 5

# TEST ranked average
blend_test = (
    rankdata(test_hist.ravel()) +
    rankdata(test_xgbm.ravel()) +
    rankdata(test_lgbm.ravel()) +
    rankdata(test_resnet.ravel()) +
    rankdata(test_realmlp.ravel())
) / 5

# Score
score = roc_auc_score(target, blend_oof)

print(f"Ranked Blend AUC: {score:.6f}")

Ranked Blend AUC: 0.951187
